In [63]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import os

In [64]:
# Set the path to the dataset
data_dir = 'plant_images'

if not os.path.isdir(data_dir):
    raise FileNotFoundError(
        f"Dataset path not found: {data_dir}. "
        "Make sure the 'plant_images' folder exists in the workspace."
    )

In [65]:
# Create an ImageDataGenerator for data augmentation and normalization
class_counts = []
for class_name in os.listdir(data_dir):
    class_path = os.path.join(data_dir, class_name)
    if os.path.isdir(class_path):
        file_count = len([
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f))
        ])
        if file_count > 0:
            class_counts.append(file_count)

if not class_counts:
    raise FileNotFoundError(
        f"No class subfolders with images found in '{data_dir}'"
    )

min_class_count = min(class_counts)
validation_split = min(0.5, max(0.2, 1.0 / min_class_count))
print(
    f"Using validation_split={validation_split:.3f} "
    f"based on minimum class size {min_class_count}"
)

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=validation_split)

Using validation_split=0.500 based on minimum class size 2


In [66]:
# Create the training data generator
train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(150, 150), # Resize all images to 150x150
    batch_size=32,
    class_mode='categorical', # For multi-class classification
    subset='training') # Set as training data

Found 9 images belonging to 9 classes.


In [67]:
# Create the validation data generator
validation_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation') # Set as validation data

Found 9 images belonging to 9 classes.


In [68]:
# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dropout(0.5), # Add dropout to prevent overfitting
    Dense(512, activation='relu'),
    Dense(train_generator.num_classes, activation='softmax') # The output layer has as many neurons as there are classes
])

In [69]:
# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [70]:
# Train the model
train_steps = max(1, train_generator.samples // train_generator.batch_size)
val_steps = max(1, validation_generator.samples // validation_generator.batch_size)

if train_generator.samples == 0:
    raise ValueError(
        f"No training images found in '{data_dir}' with subset='training'. "
        "Verify your class subfolders and the validation_split value."
    )
if validation_generator.samples == 0:
    raise ValueError(
        f"No validation images found in '{data_dir}' with subset='validation'. "
        "Verify your class subfolders and the validation_split value."
    )

history = model.fit(
      train_generator,
      steps_per_epoch=train_steps, # Number of batches per epoch
      epochs=25,
      validation_data=validation_generator,
      validation_steps=val_steps) # Number of validation batches

Epoch 1/25


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2222 - loss: 2.2058WARNING:tensorflow:6 out of the last 8 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x000002681FC44E00> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.2222 - loss: 2.2058 - val_accuracy: 0.1111 - val_loss: 2.2218
Epoch 2/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.1111 - loss: 2.1471 - val_accuracy: 0.1111 - val_loss: 2.2578
Epoch 3/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.2222 - loss: 2.2068 - val_accuracy: 0.0000e+00 - val_loss: 2.2384
Epoch 4/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step - accuracy: 0.2222 - loss: 2.1379 - val_accuracy: 0.2222 - val_loss: 2.3464
Epoch 5/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.2222 - loss: 2.2429 - val_accuracy: 0.1111 - val_loss: 2.3257
Epoch 6/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 664ms/step - accuracy: 0.0000e+00 - loss: 2.2914 - val_accuracy: 0.1111 - val_loss: 2.2149
Epoch 7/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 0.0000e+00 - loss: 2.2827 - val_accuracy: 0.0000e+00 - val_loss: 2.2913
Epoch 8/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step - accuracy: 0.0000e+00 - loss: 2.4038 - val_accuracy: 0.1111 - val

In [71]:
# Save the model
model.save('model/image_classification.h5')